In [ ]:
import subprocess
import numpy as np
from numpy import linalg as LA
import matplotlib
import json

matplotlib.use('TkAgg')
import matplotlib.pyplot as plt

In [ ]:
def get_data(bound, count, output="output.tmp",
             method="euler", summation="simple", binary=True,
             omega=1.0, x0=1.0, v0=1.0):
    settings = {
        "output": output,
        "method": method,
        "summation": summation,
        "binary": binary,
        "bound": bound,
        "count": count,
        "omega": omega,
        "x0": x0,
        "v0": v0
    }
    with open("settings.json", "w") as f:
        json.dump(settings, f)

    res = subprocess.run(["./solver", "settings.json"])
    if binary:
        data = np.fromfile(output, dtype="<f8").reshape(-1, 3)
        data = data.T
    else:
        data = np.loadtxt(output, delimiter=" ", dtype=float)
        data = (data[:, 0], data[:, 1], data[:, 2])
    
    return data

def lin(x, y):
    n = len(x)
    
    k = np.sum(x * y) / np.sum(x**2)
    
    residuals = y - k * x
    if n > 1:
        s2 = np.sum(residuals**2) / (n - 1)
    else:
        s2 = 0.0
    
    err_k = np.sqrt(s2 / np.sum(x**2))
    
    return k, err_k

def lin_(x, y):
    xy = x*y
    k = (np.mean(xy) - np.mean(x) * np.mean(y))/\
        (np.mean(x**2) - np.mean(x)**2)
    a = np.mean(y) - k*np.mean(x)

    sqrtn = (len(xy))**0.5
    err_k = 1/sqrtn * ((np.mean(y**2) - np.mean(y)**2)/\
                       (np.mean(x**2) - np.mean(x)**2) - k**2)**0.5
    err_a = err_k*(np.mean(x**2) - np.mean(x)**2)**0.5

    return k, err_k, a, err_a

In [ ]:
def get_properties(omega, x0, v0):
    phi = np.arctan(x0 * omega / v0)
    A = v0 / np.cos(phi) / omega
    return phi, A

Properties

In [ ]:
x0 = 0.0
v0 = 1.0
omega = 1.0
phi, A = get_properties(omega, x0, v0)

def get_all(bound, count, method, summation="simple"):
    data = get_data(bound, count, method=method, summation=summation,
                    omega=omega, x0=x0, v0=v0)
    t, x, v = data
    return t, x, v

Plot

In [ ]:
method = "rk4"

bound = 100
count = 100000

t, x, v = get_all(bound, count, method, summation="kahan")
plt.grid()
plt.plot(t, x)
# plt.plot(t, A * np.sin(omega * t + phi))
plt.xlabel("$t$")
plt.ylabel(r"$x$")
plt.show()

plt.close()

Energy plot

In [ ]:
method = "rk4"

bound = 10000
count = 1000000

t, x, v = get_all(bound, count, method, summation="kahan")
plt.grid()
plt.plot(t, v * v / 2 - omega * omega * np.cos(x))
plt.xlabel("$t$")
plt.ylabel(r"$E$")
plt.show()

plt.close()

Phase diagrams

In [ ]:
method = "rk4"

bound = 100
count = 50000

t, x, v = get_all(bound, count, method)
plt.grid()
plt.plot(x, v)
plt.plot(A * np.sin(omega * t + phi), A * omega * np.cos(omega * t + phi))
plt.xlabel("$x$")
plt.ylabel(r"$\dot{x}$")

plt.show()
plt.close()

Bounds of methods

Find approx by energy

In [ ]:
summation = "kahan"
for m in ["euler", "bw_eu", "heun", "rk4"]:
    bound = 10.0
    dots = 40

    count = 1.25
    mult = np.sqrt(2)

    times = []
    errors = []
    for i in range(dots):
        dt = bound/count
        times.append(dt)

        t, x, v = get_all(bound, count, m, summation=summation)
        x = x[-1]
        v = v[-1]
        errors.append(np.abs(v0*v0/2 - omega**2 * np.cos(x0) -
                      v*v/2 + omega**2 * np.cos(x)))

        count *= mult

    times = np.log(times)
    errors = np.log(errors)
    k, err_k, a, err_a = lin_(times, errors)
    print(m, k)
    plt.plot(times, errors, label=m)

plt.xlabel(r"$\ln{\Delta t}$")
plt.ylabel(r"$\ln{\text{Error}}$")
plt.legend()
plt.grid()
plt.show()